# Iterative SVD

We first install the necessary packages.

In [1]:
pip install scikit-surprise

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

## Temporal Splitting on Newest Data

We split the data temporally. In other words, we first find the 100000 most recent ratings. We then reserve the newest 20000 of the 100000 ratings as the test set, while the remaining 80000 is used as the training set.

In [3]:
full = pd.read_csv('data.csv').drop(columns=['Unnamed: 0'])[-100000:] # data.csv contains the 1000000 newest ratings sorted by rating date from the oldest to the newest
full # full contains the 100000 newest ratings

,movie_id,user_id,rating,date,Year,Title
100000,13673,1699005,5,2005-12-30,1995,Toy Story
100001,16047,24892,4,2005-12-30,2005,Because of Winn-Dixie
100002,4159,814572,3,2005-12-30,2002,Barbershop
100003,8181,2073708,2,2005-12-30,1995,The Net
100004,4067,1672834,4,2005-12-30,1976,Silver Streak
...,...,...,...,...,...,...
199995,8993,2183787,4,2005-12-31,2005,Family Guy Presents: Stewie Griffin: The Untol...
199996,7430,258170,4,2005-12-31,2001,Six Feet Under: Season 1
199997,8467,1534359,5,2005-12-31,1996,Eraser
199998,10168,2543295,2,2005-12-31,2003,The League of Extraordinary Gentlemen


In [4]:
train = full[:80000] # the 80000 ratings used in training
train

,movie_id,user_id,rating,date,Year,Title
100000,13673,1699005,5,2005-12-30,1995,Toy Story
100001,16047,24892,4,2005-12-30,2005,Because of Winn-Dixie
100002,4159,814572,3,2005-12-30,2002,Barbershop
100003,8181,2073708,2,2005-12-30,1995,The Net
100004,4067,1672834,4,2005-12-30,1976,Silver Streak
...,...,...,...,...,...,...
179995,13629,1318034,4,2005-12-31,1951,Alice in Wonderland
179996,17324,1719503,4,2005-12-31,2005,Hitch
179997,17324,22846,5,2005-12-31,2005,Hitch
179998,3860,1799620,2,2005-12-31,2003,Bruce Almighty


In [14]:
test = full[-20000:] # the 20000 ratings used for testing
test

,movie_id,user_id,rating,date,Year,Title
180000,16445,1250138,2,2005-12-31,2003,"House of 1,000 Corpses"
180001,6850,714682,1,2005-12-31,2005,Lords of Dogtown
180002,3441,859907,2,2005-12-31,2005,Kicking & Screaming
180003,10748,2373473,3,2005-12-31,1987,Hamburger Hill
180004,5496,1678873,2,2005-12-31,2004,"I, Robot"
...,...,...,...,...,...,...
199995,8993,2183787,4,2005-12-31,2005,Family Guy Presents: Stewie Griffin: The Untol...
199996,7430,258170,4,2005-12-31,2001,Six Feet Under: Season 1
199997,8467,1534359,5,2005-12-31,1996,Eraser
199998,10168,2543295,2,2005-12-31,2003,The League of Extraordinary Gentlemen


In [ ]:
len(train['movie_id'].unique()) # number of movies in training set

7648

In [ ]:
len(train['user_id'].unique()) # number of users in training set

13245

#### Further Split of Train and Validation Sets

We use the validation set approach to select the hyperparameters for each of the models. In this case, we further divide the training set for the models into a non-validation set of size 64000 (used for training in the validation set approach) and a validation set of size 16000 (used for testing in the validation set approach).

In [7]:
non_validation = train[:64000]
non_validation # used for training when selecting hyperparameters

,movie_id,user_id,rating,date,Year,Title
100000,13673,1699005,5,2005-12-30,1995,Toy Story
100001,16047,24892,4,2005-12-30,2005,Because of Winn-Dixie
100002,4159,814572,3,2005-12-30,2002,Barbershop
100003,8181,2073708,2,2005-12-30,1995,The Net
100004,4067,1672834,4,2005-12-30,1976,Silver Streak
...,...,...,...,...,...,...
163995,5805,573429,3,2005-12-31,1998,Mr. Show: Season 4
163996,12966,372124,2,2005-12-31,2004,The Aviator
163997,9134,2249750,2,2005-12-31,2004,A Day Without a Mexican
163998,8312,923517,3,2005-12-31,1995,Jury Duty


In [8]:
validation = train[64000:]
validation # used for testing when selecting hyperparameters

,movie_id,user_id,rating,date,Year,Title
164000,8928,1655500,4,2005-12-31,2002,Snow Dogs
164001,3017,2448705,4,2005-12-31,1985,Cocoon
164002,3685,1885503,3,2005-12-31,2000,Scooby-Doo's Spookiest Tales
164003,6610,1971298,2,2005-12-31,1965,The Pawnbroker
164004,7625,2524708,3,2005-12-31,1977,Young Lady Chatterley
...,...,...,...,...,...,...
179995,13629,1318034,4,2005-12-31,1951,Alice in Wonderland
179996,17324,1719503,4,2005-12-31,2005,Hitch
179997,17324,22846,5,2005-12-31,2005,Hitch
179998,3860,1799620,2,2005-12-31,2003,Bruce Almighty


## compute_rmse Function

We define a compute_rmse function that takes in the predicted output and actual output, so that we can conveniently compute the RMSE of our predictions.

In [9]:
# Takes in two NumPy arrays
def compute_rmse(predicted, actual):
    return np.sqrt(np.mean((predicted - actual) * (predicted - actual)))

## Model Training and Testing

We implement the Iterative SVD algorithm in the IterativeSVD class below.

In [10]:
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import svds

class IterativeSVD:
    def __init__(self, k=10, num_epochs=10):
        # Initialize instance variables
        self.k = k
        self.num_epochs = num_epochs
        self.R = None
        self.global_mean = 0
        self.user_map = None
        self.movie_map = None
        self.user_means = []
        self.movie_means = []

    def fit(self, X, y):
        # Map user_id and movie_id to indices
        users = X['user_id'].unique()
        movies = X['movie_id'].unique()
        user_map = {user_id: idx for idx, user_id in enumerate(users)}
        movie_map = {movie_id: idx for idx, movie_id in enumerate(movies)}
        self.user_map = user_map
        self.movie_map = movie_map
        self.global_mean = np.mean(y)

        for u in users:
            self.user_means.append(np.mean(y[X['user_id'] == u]))
        for m in movies:
            self.movie_means.append(np.mean(y[X['movie_id'] == m]))
        self.user_means = np.array(self.user_means)
        self.movie_means = np.array(self.movie_means)

        # Create the sparse ratings matrix to save memory
        num_users = len(users)
        num_movies = len(movies)
        R = lil_matrix((num_users, num_movies))
        for (user, movie, rating) in zip(X['user_id'], X['movie_id'], y):
            R[user_map[user], movie_map[movie]] = rating - 0.5 * self.user_means[user_map[user]] - 0.5 * self.movie_means[movie_map[movie]]
        R = R.tocsr()

        # Main loop
        for epoch in range(self.num_epochs):
            # Perform SVD
            U, sigma, VT = svds(R, k=self.k)
            sigma = np.diag(sigma)

            # Reconstruct sparse matrix
            R_reconstructed = csr_matrix(U) @ csr_matrix(sigma) @ csr_matrix(VT)

            # Update the non-zero entries
            mask_non_zero = R != 0
            mask_zero = R == 0
            R = R.multiply(mask_non_zero) + R_reconstructed.multiply(mask_zero)
            self.R = R

            # Logging curent progress
            print(f"Epoch {epoch + 1} completed.")


    def predict(self, X):
        # Get the corresponding index for each user id and each movie_id
        user_indices = np.array([self.user_map.get(u, -1) for u in X['user_id']])
        item_indices = np.array([self.movie_map.get(i, -1) for i in X['movie_id']])

        # Initialize predictions
        predictions = np.full(len(X), self.global_mean)  # Set global mean as default prediction value
        valid_mask = (user_indices != -1) & (item_indices != -1)

        # Loop to predict each data point
        for idx in np.where(valid_mask)[0]:
            user_idx = user_indices[idx]
            item_idx = item_indices[idx]
            predictions[idx] = (
                self.R[user_idx, item_idx]
                + 0.5 * self.user_means[user_idx]
                + 0.5 * self.movie_means[item_idx]
            )

        return predictions


    def get_full_predictions(self):
        return self.R.toarray() + self.global_mean


Below is the code to select the best value of K. We did not run this piece of code due to limited computing power.

In [11]:
'''best_rmse = np.inf
best_params = None

# Candidate hyperparameter values
K_list = [10, 20, 100]

# Main loop
for K in K_list:
  model = IterativeSVD(K, 3)
  model.fit(non_validation.drop(columns=['rating']), non_validation['rating'])
  rmse = compute_rmse(validation['rating'].to_numpy(), model.predict(validation))
  if (rmse < best_rmse):
    best_rmse = rmse
    best_params = [K]'''

"best_rmse = np.inf\nbest_params = None\n\n# Candidate hyperparameter values\nK_list = [10, 20, 100]\n\n# Main loop\nfor K in K_list:\n  model = IterativeSVD(K, 3)\n  model.fit(non_validation.drop(columns=['rating']), non_validation['rating'])\n  rmse = compute_rmse(validation['rating'].to_numpy(), model.predict(validation))\n  if (rmse < best_rmse):\n    best_rmse = rmse\n    best_params = [K]"

We train an Iterative SVD model with $K=2$ and 3 epochs, and then evaluate its test RMSE.

In [12]:
# Model training
iter_svd_100000 = IterativeSVD(2,3)
iter_svd_100000.fit(train[['movie_id', 'user_id']], train['rating'])

/var/folders/f0/94sztnyj0hsgk3msr8gkkvpm0000gq/T/ipykernel_90203/940379082.py:52: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient. Try using != instead.
  mask_zero = R == 0


Epoch 1 completed.
Epoch 2 completed.
Epoch 3 completed.


In [15]:
# Evaluate the test RMSE
compute_rmse(test['rating'].to_numpy(), iter_svd_100000.predict(test))

np.float64(0.9693799827903011)